<div style="background:#1e1e2e; padding:16px 20px; border-radius:8px; font-family:monospace; color:#cdd6f4; line-height:1.8"><p style="margin:0 0 8px 0; color:#cba6f7; font-weight:bold; font-size:1.05em;">✅ Session 7E — GIL &amp; Concurrency · Solutions</p><p style="margin:0;">Worked, runnable solutions for the 12 <strong>Exercises</strong> and 8 <strong>Code Challenges</strong>. <strong>Notes:</strong> timings vary per run; async cells use <code>asyncio.run()</code> (in Jupyter use top-level <code>await main()</code> instead); multiprocessing cells (E10 / C7) must run as a <strong>.py script</strong> with the <code>if __name__ == "__main__"</code> guard.</p></div>

### Exercises — Solutions

In [ ]:
import threading, queue, asyncio, time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

In [ ]:
# E1 — two functions concurrently
res = []
def work(x, out): out.append(x * x)
ts = [threading.Thread(target=work, args=(n, res)) for n in [1, 2, 3]]
for t in ts: t.start()
for t in ts: t.join()
print(sorted(res))                    # [1, 4, 9]

In [ ]:
# E2 — ThreadPoolExecutor.map (order preserved)
with ThreadPoolExecutor() as ex:
    print(list(ex.map(lambda x: x * 2, [1, 2, 3])))   # [2, 4, 6]

In [ ]:
# E3 — pick the tool
def pick_tool(kind):
    return "multiprocessing" if kind == "cpu" else "threads/asyncio"

print(pick_tool("cpu"), "|", pick_tool("io"))

In [ ]:
# E4 — I/O overlap (timing varies)
def io(): time.sleep(0.05)
t0 = time.perf_counter(); io(); io(); ser = time.perf_counter() - t0
t0 = time.perf_counter()
ts = [threading.Thread(target=io) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
thr = time.perf_counter() - t0
print(f"serial={ser*1000:.0f}ms  threaded={thr*1000:.0f}ms  (threads overlap the sleeps)")

In [ ]:
# E5 — fix a race with a Lock
counter = 0
lock = threading.Lock()
def inc():
    global counter
    for _ in range(50000):
        with lock:
            counter += 1
ts = [threading.Thread(target=inc) for _ in range(2)]
for t in ts: t.start()
for t in ts: t.join()
print(counter)                        # 100000 (exact, thanks to the lock)

In [ ]:
# E6 — producer/consumer with a queue + sentinel
q = queue.Queue(); out = []
def producer():
    for i in range(5): q.put(i)
    q.put(None)
def consumer():
    while True:
        item = q.get()
        if item is None: break
        out.append(item * 10)
p = threading.Thread(target=producer); c = threading.Thread(target=consumer)
p.start(); c.start(); p.join(); c.join()
print(out)                            # [0, 10, 20, 30, 40]

In [ ]:
# E7 — collect results as they finish
with ThreadPoolExecutor() as ex:
    futs = [ex.submit(lambda x: x * x, n) for n in [1, 2, 3]]
    print(sorted(f.result() for f in as_completed(futs)))   # [1, 4, 9]

In [ ]:
# E8 — asyncio.gather two coroutines
async def task(n):
    await asyncio.sleep(0.01)
    return n * n
async def main():
    return await asyncio.gather(task(1), task(2), task(3))
print(asyncio.run(main()))            # [1, 4, 9]   (in Jupyter: await main())

In [ ]:
# E9 — gather over N fetches (total ~= slowest, not sum)
async def fetch(i):
    await asyncio.sleep(0.03)
    return i
async def main():
    return await asyncio.gather(*[fetch(i) for i in range(5)])
t0 = time.perf_counter(); r = asyncio.run(main()); dt = time.perf_counter() - t0
print(r, f"| ~{dt*1000:.0f}ms (not 150ms)")

In [ ]:
# E10 — CPU-bound speedup with ProcessPoolExecutor (RUN AS A SCRIPT)
def cpu_sum(n):
    s = 0
    for i in range(n): s += i
    return s

if __name__ == "__main__":            # required: spawn re-imports the module
    with ProcessPoolExecutor() as ex:
        print(list(ex.map(cpu_sum, [1000, 2000, 3000])))   # [499500, 1999000, 4498500]

In [ ]:
# E11 — thread-safe memoize (computes once)
def make_cache():
    cache, lock, calls = {}, threading.Lock(), [0]
    def get(k, compute):
        with lock:
            if k not in cache:
                calls[0] += 1
                cache[k] = compute(k)
            return cache[k]
    return get, calls

get, calls = make_cache()
print(get(4, lambda x: x*x), get(4, lambda x: x*x), "| computed", calls[0])   # 16 16 | computed 1

In [ ]:
# E12 — worker pool draining a queue with N sentinels
q = queue.Queue(); out = []; olock = threading.Lock()
def worker():
    while True:
        item = q.get()
        if item is None: break
        with olock: out.append(item * item)
N = 3
workers = [threading.Thread(target=worker) for _ in range(N)]
for w in workers: w.start()
for i in range(6): q.put(i)
for _ in range(N): q.put(None)        # one sentinel per worker
for w in workers: w.join()
print(sorted(out))                    # [0, 1, 4, 9, 16, 25]

### Code Challenges — Solutions

In [ ]:
# C1 — parallel map with a thread pool
def square(x): return x * x
with ThreadPoolExecutor(max_workers=3) as ex:
    print(list(ex.map(square, range(3))))   # [0, 1, 4]

In [ ]:
# C2 — concurrent fetches returned in order
def fetch(u): time.sleep(0.01); return f"data:{u}"
with ThreadPoolExecutor() as ex:
    print(list(ex.map(fetch, ["a", "b", "c"])))   # ['data:a', 'data:b', 'data:c']

In [ ]:
# C3 — thread-safe Counter (Lock)
class Counter:
    def __init__(self): self._v = 0; self._lock = threading.Lock()
    def inc(self):
        with self._lock: self._v += 1
    @property
    def value(self): return self._v

c = Counter()
ts = [threading.Thread(target=lambda: [c.inc() for _ in range(10000)]) for _ in range(4)]
for t in ts: t.start()
for t in ts: t.join()
print(c.value)                        # 40000

In [ ]:
# C4 — asyncio.gather in order
async def dbl(n):
    await asyncio.sleep(0.01)
    return n * 2
async def main():
    return await asyncio.gather(*[dbl(i) for i in range(4)])
print(asyncio.run(main()))            # [0, 2, 4, 6]

In [ ]:
# C5 — timeout a slow coroutine
async def slow():
    await asyncio.sleep(1)
    return "done"
async def main():
    try:
        return await asyncio.wait_for(slow(), timeout=0.05)
    except asyncio.TimeoutError:
        return "timeout"
print(asyncio.run(main()))            # timeout

In [ ]:
# C6 — cap concurrency with a Semaphore (never more than 2 active)
sem = threading.Semaphore(2)
active, mx, lk = [], [0], threading.Lock()
def task():
    with sem:
        with lk:
            active.append(1); mx[0] = max(mx[0], sum(active))
        time.sleep(0.02)
        with lk: active.pop()
ts = [threading.Thread(target=task) for _ in range(6)]
for t in ts: t.start()
for t in ts: t.join()
print("max concurrent:", mx[0], "(<= 2)")

In [ ]:
# C7 — CPU-bound parallel map with ProcessPoolExecutor (RUN AS A SCRIPT)
def sq(x): return x * x            # top-level, picklable

if __name__ == "__main__":
    with ProcessPoolExecutor(max_workers=4) as ex:
        print(list(ex.map(sq, range(6))))   # [0, 1, 4, 9, 16, 25]

In [ ]:
# C8 — async producer/consumer with asyncio.Queue
async def main():
    q = asyncio.Queue(); out = []
    async def prod():
        for i in range(5): await q.put(i)
        await q.put(None)
    async def cons():
        while True:
            item = await q.get()
            if item is None: break
            out.append(item * 10)
    await asyncio.gather(prod(), cons())
    return out
print(asyncio.run(main()))            # [0, 10, 20, 30, 40]